# 13 OOD / Severity-Blocked Validation

This notebook runs the additive protection-event parser recovery and the full 30/60/90/120 s severity-blocked, extrapolation, matched-cohort, sensitivity, and nearest-severity-gap audit. All splits use complete `sample_id` trajectories and all preprocessing is fit on train only.

In [1]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src' / 'ood_validation.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('NPP-Guard project root not found')
sys.path.insert(0, str(PROJECT_ROOT))
from src.ood_validation import run_experiment

summary = run_experiment(PROJECT_ROOT)
summary['gate']

{'severity_blocked': {'available': True,
  'macro_f1_120': 0.5931285851575706,
  'balanced_accuracy_120': 0.9212962962962963,
  'macro_f1_120_exceeds_60_90_by_mean': True,
  'macro_f1_120_exceeds_60_90_by_ci': True,
  'balanced_accuracy_120_exceeds_60_90_by_mean': True},
 'severity_extrapolation': {'available': True,
  'macro_f1_120': 0.48258843494350745,
  'balanced_accuracy_120': 0.8125,
  'macro_f1_120_exceeds_60_90_by_mean': True,
  'macro_f1_120_exceeds_60_90_by_ci': True,
  'balanced_accuracy_120_exceeds_60_90_by_mean': True},
 'ood_120_macro_f1_mean_at_least_0_50': False,
 'ood_120_balanced_accuracy_mean_at_least_0_50': True,
 'ood_120_ci_advantage_over_controls': True,
 'all_ood_recall_classes_have_support_ge_5': True,
 'all_ood_recall_classes_mean_at_least_0_80': False,
 'decision': 'B_REQUIRE_FURTHER_DATA_OR_VALIDATION'}

In [2]:
assert summary['ood_matched_cohort_count'] > 20
assert summary['assertions']['trajectory_level_assignments']
assert summary['assertions']['train_only_model_fit']
assert summary['assertions']['same_ood_cohort_for_60_90_120']
assert summary['parser_recovery']['trajectory_total'] == 1211
print('FULL OOD SEVERITY-BLOCKED VALIDATION PASSED')
print('decision:', summary['gate']['decision'])
print('matched cohort:', summary['ood_matched_cohort_count'])

FULL OOD SEVERITY-BLOCKED VALIDATION PASSED
decision: B_REQUIRE_FURTHER_DATA_OR_VALIDATION
matched cohort: 505


In [3]:
import pandas as pd
pd.DataFrame(summary['metric_summary_120s'])[[
    'split_type', 'macro_f1_fixed_12_mean', 'macro_f1_fixed_12_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std'
]]

,split_type,macro_f1_fixed_12_mean,macro_f1_fixed_12_std,balanced_accuracy_mean,balanced_accuracy_std
0,random,0.657251,0.009479,0.983896,0.016421
1,severity_blocked,0.593129,0.063816,0.921296,0.068512
2,severity_extrapolation,0.482588,0.110099,0.812500,0.088388
